# Notebook 03 — Question Segmentation and Structuring

## Purpose
Convert extracted exam/memo text into a structured question table.

## Scope
- Detect main questions (`QUESTION 1`, `QUESTION 2`, ...)
- Detect subquestions (`1.1`, `1.2`, ...)
- Capture mark allocations where present
- Preserve source document linkage

## Out of scope
- CAPS topic classification
- Topic modelling
- Misconception mining
- Forecasting

## Primary output
`data/processed/questions/question_structure.csv`

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
META_DIR = DATA_DIR / "metadata"
PROCESSED_DIR = DATA_DIR / "processed"

EXTRACTED_DIR = PROCESSED_DIR / "extracted_text"
OCR_TEXT_DIR = PROCESSED_DIR / "ocr" / "text"
QUESTIONS_DIR = PROCESSED_DIR / "questions"
QUESTIONS_DIR.mkdir(parents=True, exist_ok=True)

register_path = META_DIR / "exam_document_register.csv"
extraction_register_path = META_DIR / "document_extraction_register.csv"
ocr_register_path = META_DIR / "document_ocr_register.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("QUESTIONS_DIR:", QUESTIONS_DIR)

PROJECT_ROOT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
QUESTIONS_DIR: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\questions


In [4]:
register_df = pd.read_csv(register_path)
extraction_df = pd.read_csv(extraction_register_path)

if ocr_register_path.exists():
    ocr_df = pd.read_csv(ocr_register_path)
else:
    ocr_df = pd.DataFrame()

print("Master register rows:", len(register_df))
print("Extraction rows:", len(extraction_df))
print("OCR rows:", len(ocr_df))

Master register rows: 206
Extraction rows: 14
OCR rows: 7


In [5]:
# Merge metadata from master register
meta_cols = ["document_id", "year", "paper", "document_type"]
meta_cols = [c for c in meta_cols if c in register_df.columns]

usable = usable.merge(
    register_df[meta_cols].drop_duplicates("document_id"),
    on="document_id",
    how="left"
)

print("Usable documents for segmentation:", len(usable))
print("Columns:", usable.columns.tolist())

show_cols = [
    c for c in [
        "document_id",
        "year",
        "paper",
        "document_type",
        "extraction_quality",
        "resolved_text_file",
    ]
    if c in usable.columns
]

display(usable[show_cols])

Usable documents for segmentation: 14
Columns: ['document_id', 'file_name', 'pdf_path', 'text_file', 'page_count', 'character_count', 'average_chars_per_page', 'pages_with_text', 'text_page_ratio', 'extraction_quality', 'ocr_required', 'extraction_status', 'extraction_date', 'resolved_text_file', 'year', 'paper', 'document_type']


,document_id,year,paper,document_type,extraction_quality,resolved_text_file
0,2023_nov_p1_exam_maths,2023,P1,exam,ocr_good,data\processed\ocr\text\2023_nov_p1_exam_maths...
1,2023_nov_p1_memo_maths,2023,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...
2,2023_nov_p2_exam_maths,2023,P2,exam,ocr_good,data\processed\ocr\text\2023_nov_p2_exam_maths...
3,2023_nov_p2_memo_maths,2023,P2,memo,good,data\processed\extracted_text\NSC_Mathematics_...
4,2024_nov_p1_exam_maths,2024,P1,exam,ocr_good,data\processed\ocr\text\2024_nov_p1_exam_maths...
5,2024_nov_p1_memo_maths,2024,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...
6,2024_nov_p2_exam_maths,2024,P2,exam,ocr_good,data\processed\ocr\text\2024_nov_p2_exam_maths...
7,2024_nov_p2_memo_maths,2024,P2,memo,good,data\processed\extracted_text\NSC_Mathematics_...
8,2025_nov_p1_exam_maths,2025,P1,exam,ocr_good,data\processed\ocr\text\2025_nov_p1_exam_maths...
9,2025_nov_p1_memo_maths,2025,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...


In [6]:
def resolve_text_path(row):
    if pd.notna(row.get("text_file")):
        p = PROJECT_ROOT / str(row["text_file"])
        if p.exists():
            return str(p.relative_to(PROJECT_ROOT))

    ocr_candidate = OCR_TEXT_DIR / f"{row['document_id']}_ocr.txt"
    if ocr_candidate.exists():
        return str(ocr_candidate.relative_to(PROJECT_ROOT))

    return None

usable["resolved_text_file"] = usable.apply(resolve_text_path, axis=1)
usable = usable[usable["resolved_text_file"].notna()].copy()

In [8]:
# ============================================================
# USABLE DOCUMENTS FOR SEGMENTATION
# ============================================================

usable = extraction_df[
    extraction_df["extraction_status"] == "extracted"
].copy()

usable = usable[
    usable["extraction_quality"].isin(["good", "partial", "ocr_good"])
].copy()

def resolve_text_path(row):
    # Prefer existing text_file if present
    if pd.notna(row.get("text_file")):
        p = PROJECT_ROOT / str(row["text_file"])
        if p.exists():
            return str(p.relative_to(PROJECT_ROOT))

    # Prefer OCR file by convention
    ocr_candidate = OCR_TEXT_DIR / f"{row['document_id']}_ocr.txt"
    if ocr_candidate.exists():
        return str(ocr_candidate.relative_to(PROJECT_ROOT))

    return None

usable["resolved_text_file"] = usable.apply(resolve_text_path, axis=1)
usable = usable[usable["resolved_text_file"].notna()].copy()

# Merge metadata from master register
for col in ["year", "paper", "document_type"]:
    if col not in usable.columns and col in register_df.columns:
        usable = usable.merge(
            register_df[["document_id", col]].drop_duplicates("document_id"),
            on="document_id",
            how="left"
        )

print("Usable documents for segmentation:", len(usable))
print("Available columns:", usable.columns.tolist())

show_cols = [
    c for c in [
        "document_id",
        "year",
        "paper",
        "document_type",
        "extraction_quality",
        "resolved_text_file",
    ]
    if c in usable.columns
]

display(usable[show_cols])

Usable documents for segmentation: 14
Available columns: ['document_id', 'file_name', 'pdf_path', 'text_file', 'page_count', 'character_count', 'average_chars_per_page', 'pages_with_text', 'text_page_ratio', 'extraction_quality', 'ocr_required', 'extraction_status', 'extraction_date', 'resolved_text_file', 'year', 'paper', 'document_type']


,document_id,year,paper,document_type,extraction_quality,resolved_text_file
0,2023_nov_p1_exam_maths,2023,P1,exam,ocr_good,data\processed\ocr\text\2023_nov_p1_exam_maths...
1,2023_nov_p1_memo_maths,2023,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...
2,2023_nov_p2_exam_maths,2023,P2,exam,ocr_good,data\processed\ocr\text\2023_nov_p2_exam_maths...
3,2023_nov_p2_memo_maths,2023,P2,memo,good,data\processed\extracted_text\NSC_Mathematics_...
4,2024_nov_p1_exam_maths,2024,P1,exam,ocr_good,data\processed\ocr\text\2024_nov_p1_exam_maths...
5,2024_nov_p1_memo_maths,2024,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...
6,2024_nov_p2_exam_maths,2024,P2,exam,ocr_good,data\processed\ocr\text\2024_nov_p2_exam_maths...
7,2024_nov_p2_memo_maths,2024,P2,memo,good,data\processed\extracted_text\NSC_Mathematics_...
8,2025_nov_p1_exam_maths,2025,P1,exam,ocr_good,data\processed\ocr\text\2025_nov_p1_exam_maths...
9,2025_nov_p1_memo_maths,2025,P1,memo,good,data\processed\extracted_text\NSC_Mathematics_...


In [9]:
corpus_rows = []

for _, row in usable.iterrows():
    text_path = PROJECT_ROOT / row["resolved_text_file"]
    text = text_path.read_text(encoding="utf-8", errors="ignore")

    corpus_rows.append({
        "document_id": row["document_id"],
        "year": row.get("year"),
        "paper": row.get("paper"),
        "document_type": row.get("document_type"),
        "source_text_file": row["resolved_text_file"],
        "extraction_quality": row.get("extraction_quality"),
        "text": text,
        "char_count": len(text),
    })

corpus_df = pd.DataFrame(corpus_rows)
print("Corpus documents:", len(corpus_df))
display(corpus_df[["document_id", "document_type", "char_count", "extraction_quality"]])

Corpus documents: 14


,document_id,document_type,char_count,extraction_quality
0,2023_nov_p1_exam_maths,exam,8423,ocr_good
1,2023_nov_p1_memo_maths,memo,13478,good
2,2023_nov_p2_exam_maths,exam,9620,ocr_good
3,2023_nov_p2_memo_maths,memo,19359,good
4,2024_nov_p1_exam_maths,exam,9737,ocr_good
5,2024_nov_p1_memo_maths,memo,14185,good
6,2024_nov_p2_exam_maths,exam,10389,ocr_good
7,2024_nov_p2_memo_maths,memo,21400,good
8,2025_nov_p1_exam_maths,exam,8995,ocr_good
9,2025_nov_p1_memo_maths,memo,16147,good


In [10]:
def clean_for_segmentation(text: str) -> str:
    """Conservative cleaning that preserves question structure."""
    if not text:
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Fix common OCR joins
    text = re.sub(r"QUESTION\s*\n\s*(\d+)", r"QUESTION \1", text, flags=re.IGNORECASE)
    text = re.sub(r"\bQ\s*UESTION\b", "QUESTION", text, flags=re.IGNORECASE)

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

corpus_df["clean_text"] = corpus_df["text"].apply(clean_for_segmentation)
print("Cleaning complete")

Cleaning complete


In [11]:
QUESTION_SPLIT_RE = re.compile(
    r"(?=(?:^|\n)\s*QUESTION\s+(\d+)\b)",
    flags=re.IGNORECASE
)

def split_main_questions(text: str):
    """
    Split text into blocks by QUESTION n markers.
    Returns list of dicts: question_number, question_block
    """
    if not text:
        return []

    matches = list(re.finditer(r"(?:^|\n)\s*QUESTION\s+(\d+)\b", text, flags=re.IGNORECASE))
    if not matches:
        return []

    blocks = []
    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        blocks.append({
            "question_number": qnum,
            "question_block": block,
        })
    return blocks

# Test on first exam-like document
sample = corpus_df.iloc[0]
sample_blocks = split_main_questions(sample["clean_text"])
print(sample["document_id"], "→ main questions found:", len(sample_blocks))
for b in sample_blocks[:5]:
    print(" Q", b["question_number"], "| chars:", len(b["question_block"]))

2023_nov_p1_exam_maths → main questions found: 10
 Q 1 | chars: 391
 Q 2 | chars: 446
 Q 3 | chars: 607
 Q 4 | chars: 606
 Q 5 | chars: 749


In [12]:
SUBQ_RE = re.compile(
    r"(?m)^\s*(\d+\.\d+(?:\.\d+)?)\s+"
)

MARKS_RE = re.compile(
    r"\[\s*(\d+)\s*\]"
)

def extract_marks(text: str):
    marks = [int(x) for x in MARKS_RE.findall(text or "")]
    return marks

def split_subquestions(question_number: int, block: str):
    """
    Split a main question block into subquestion units when possible.
    Fallback: one record for the whole question.
    """
    matches = list(SUBQ_RE.finditer(block))
    if not matches:
        marks = extract_marks(block)
        return [{
            "subquestion": None,
            "marks": marks[-1] if marks else None,
            "all_marks_found": marks,
            "question_text": block.strip(),
        }]

    records = []
    for i, m in enumerate(matches):
        subq = m.group(1)
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(block)
        sub_text = block[start:end].strip()
        marks = extract_marks(sub_text)
        records.append({
            "subquestion": subq,
            "marks": marks[-1] if marks else None,
            "all_marks_found": marks,
            "question_text": sub_text,
        })
    return records

# quick test
if sample_blocks:
    demo = split_subquestions(sample_blocks[0]["question_number"], sample_blocks[0]["question_block"])
    print("Subquestion records in first main question:", len(demo))
    display(pd.DataFrame(demo)[["subquestion", "marks"]])

Subquestion records in first main question: 7


,subquestion,marks
0,1.1,NaN
1,11.1,NaN
2,1.1.2,NaN
3,1.1.3,NaN
4,1.1.4,NaN
5,1.2,NaN
6,1.3,24.0


In [13]:
import re

# Subquestions like 1.1 / 1.1.1 / 2.3 only at line starts
SUBQ_RE = re.compile(
    r"(?m)^\s*(\d{1,2}\.\d{1,2}(?:\.\d{1,2})?)\s+"
)

# Common mark forms in NSC papers / OCR
MARKS_RE = re.compile(
    r"(?:\[\s*(\d{1,2})\s*\]|\(\s*(\d{1,2})\s*\)|\b(\d{1,2})\s*marks?\b)",
    flags=re.IGNORECASE
)

def extract_marks(text: str):
    vals = []
    for a, b, c in MARKS_RE.findall(text or ""):
        for x in (a, b, c):
            if x:
                vals.append(int(x))
    # keep plausible exam marks only
    return [m for m in vals if 1 <= m <= 20]


def split_subquestions(question_number: int, block: str):
    matches = list(SUBQ_RE.finditer(block))

    # Keep only subquestions that belong to this main question
    # e.g. for Q1 accept 1.1, 1.2, 1.1.1 — reject 11.1, 2.1
    filtered = []
    for m in matches:
        subq = m.group(1)
        main = int(subq.split(".")[0])
        if main == question_number:
            filtered.append(m)

    if not filtered:
        marks = extract_marks(block)
        return [{
            "subquestion": None,
            "marks": marks[-1] if marks else None,
            "all_marks_found": marks,
            "question_text": block.strip(),
        }]

    records = []
    for i, m in enumerate(filtered):
        subq = m.group(1)
        start = m.start()
        end = filtered[i + 1].start() if i + 1 < len(filtered) else len(block)
        sub_text = block[start:end].strip()
        marks = extract_marks(sub_text)

        records.append({
            "subquestion": subq,
            "marks": marks[-1] if marks else None,
            "all_marks_found": marks,
            "question_text": sub_text,
        })

    return records

In [14]:
sample = corpus_df.iloc[0]
blocks = split_main_questions(sample["clean_text"])
print("Main questions:", [b["question_number"] for b in blocks])

if blocks:
    demo = split_subquestions(blocks[0]["question_number"], blocks[0]["question_block"])
    display(pd.DataFrame(demo)[["subquestion", "marks", "all_marks_found"]])
    print("\nSample text:\n")
    print(demo[0]["question_text"][:800])

Main questions: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,subquestion,marks,all_marks_found
0,1.1,3,[3]
1,1.1.2,4,[4]
2,1.1.3,4,[4]
3,1.1.4,4,[4]
4,1.2,5,[5]
5,1.3,4,[4]



Sample text:

1.1 Solve for x:

11.1 x +x-12=0 (3)


In [15]:
question_records = []

for _, doc in corpus_df.iterrows():
    blocks = split_main_questions(doc["clean_text"])

    if not blocks:
        question_records.append({
            "document_id": doc["document_id"],
            "year": doc["year"],
            "paper": doc["paper"],
            "document_type": doc["document_type"],
            "source_text_file": doc["source_text_file"],
            "extraction_quality": doc["extraction_quality"],
            "question_number": None,
            "subquestion": None,
            "marks": None,
            "question_text": doc["clean_text"][:2000],
            "segmentation_status": "no_question_markers",
        })
        continue

    for block in blocks:
        qn = block["question_number"]
        sub_records = split_subquestions(qn, block["question_block"])

        for j, sub in enumerate(sub_records, start=1):
            question_records.append({
                "document_id": doc["document_id"],
                "year": doc["year"],
                "paper": doc["paper"],
                "document_type": doc["document_type"],
                "source_text_file": doc["source_text_file"],
                "extraction_quality": doc["extraction_quality"],
                "question_number": qn,
                "subquestion": sub["subquestion"],
                "marks": sub["marks"],
                "marks_all_found": json.dumps(sub["all_marks_found"]),
                "question_text": sub["question_text"],
                "question_char_count": len(sub["question_text"]),
                "segment_index": j,
                "segmentation_status": "ok",
            })

questions_df = pd.DataFrame(question_records)
print("Structured records:", len(questions_df))
display(questions_df.head(20))

Structured records: 417


,document_id,year,paper,document_type,source_text_file,extraction_quality,question_number,subquestion,marks,marks_all_found,question_text,question_char_count,segment_index,segmentation_status
0,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1,3.0,[3],1.1 Solve for x:\n\n11.1 x +x-12=0 (3),36.0,1.0,ok
1,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.2,4.0,[4],1.1.2 3x*-2x=6 (answers correct to TWO decimal...,58.0,2.0,ok
2,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.3,4.0,[4],1.1.3 V2xt+1 =x-1 (4),21.0,3.0,ok
3,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.1.4,4.0,[4],1.1.4 x? -3>2x (4),18.0,4.0,ok
4,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.2,5.0,[5],1.2 Solve for x and y simultaneously:\n\nx+2=2...,64.0,5.0,ok
5,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,1.0,1.3,4.0,[4],"1.3 Given: 2”*1 +2"" =3""'? 3” where m and n are...",174.0,6.0,ok
6,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,2.0,2.1,NaN,[],2.1 Given the arithmetic series: 7+ 12+17+...,45.0,1.0,ok
7,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,2.0,2.1.1,3.0,[3],"2.1.1 Determine the value of Ty, (3)",36.0,2.0,ok
8,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,2.0,2.1.2,NaN,[],"2.1.2 Calculate S,, 2)",22.0,3.0,ok
9,2023_nov_p1_exam_maths,2023,P1,exam,data\processed\ocr\text\2023_nov_p1_exam_maths...,ocr_good,2.0,2.1.3,3.0,[3],"2.1.3 Calculate the value of n for which T, =5...",52.0,4.0,ok


In [16]:
print("Segmentation status:")
print(questions_df["segmentation_status"].value_counts(dropna=False))

print("\nQuestions per document:")
print(
    questions_df.groupby("document_id")["question_number"]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

print("\nMissing marks count:", questions_df["marks"].isna().sum())
print("Records with no question number:", questions_df["question_number"].isna().sum())

# Duplicate risk check
dup_cols = ["document_id", "question_number", "subquestion"]
dupes = questions_df[questions_df.duplicated(dup_cols, keep=False)]
print("Potential duplicate keys:", len(dupes))

Segmentation status:
segmentation_status
ok                     413
no_question_markers      4
Name: count, dtype: int64

Questions per document:
document_id
2024_nov_p1_exam_maths                     12
2024_nov_p2_exam_maths                     11
2025_nov_p2_exam_maths                     11
2025_nov_p2_memo_maths                     11
2025_nov_p1_exam_maths                     11
2023_nov_p1_exam_maths                     10
2023_nov_p2_exam_maths                     10
2023_nov_p1_memo_maths                     10
2025_nov_p1_memo_maths                      1
2024_nov_p1_memo_maths                      1
2023_nov_p2_memo_maths                      0
2024_nov_p2_memo_maths                      0
caps_mathematics_gr10_12                    0
grade12_mathematics_exam_guideline_2021     0
Name: question_number, dtype: int64

Missing marks count: 164
Records with no question number: 4
Potential duplicate keys: 6


In [17]:
question_structure_path = QUESTIONS_DIR / "question_structure.csv"
questions_df.to_csv(question_structure_path, index=False)

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "documents_processed": int(corpus_df["document_id"].nunique()),
    "question_records": int(len(questions_df)),
    "records_ok": int((questions_df["segmentation_status"] == "ok").sum()),
    "records_no_markers": int((questions_df["segmentation_status"] == "no_question_markers").sum()),
    "missing_marks": int(questions_df["marks"].isna().sum()),
}

summary_path = QUESTIONS_DIR / "question_structure_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved:", question_structure_path)
print("Saved:", summary_path)
print(summary)

Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\questions\question_structure.csv
Saved: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\questions\question_structure_summary.json
{'generated_at': '2026-09-10T08:12:22.412841+00:00', 'documents_processed': 14, 'question_records': 417, 'records_ok': 413, 'records_no_markers': 4, 'missing_marks': 164}


In [18]:
print("=" * 60)
print("NOTEBOOK 03 STATUS")
print("=" * 60)
print(f"Documents processed : {summary['documents_processed']}")
print(f"Question records    : {summary['question_records']}")
print(f"OK segments         : {summary['records_ok']}")
print(f"No markers          : {summary['records_no_markers']}")
print(f"Missing marks       : {summary['missing_marks']}")
print("=" * 60)

NOTEBOOK 03 STATUS
Documents processed : 14
Question records    : 417
OK segments         : 413
No markers          : 4
Missing marks       : 164


In [19]:
# 1. Documents with no markers
print(questions_df[questions_df["segmentation_status"] == "no_question_markers"][
    ["document_id", "document_type", "extraction_quality"]
])

# 2. Questions per document
print(
    questions_df.groupby(["document_id", "document_type"])["question_number"]
    .nunique(dropna=True)
)

# 3. Mark coverage by document type
print(
    questions_df.assign(has_marks=questions_df["marks"].notna())
    .groupby("document_type")["has_marks"]
    .mean()
)

                                 document_id document_type extraction_quality
160                   2023_nov_p2_memo_maths          memo               good
259                   2024_nov_p2_memo_maths          memo               good
415                 caps_mathematics_gr10_12          caps               good
416  grade12_mathematics_exam_guideline_2021     guideline               good
document_id                              document_type
2023_nov_p1_exam_maths                   exam             10
2023_nov_p1_memo_maths                   memo             10
2023_nov_p2_exam_maths                   exam             10
2023_nov_p2_memo_maths                   memo              0
2024_nov_p1_exam_maths                   exam             12
2024_nov_p1_memo_maths                   memo              1
2024_nov_p2_exam_maths                   exam             11
2024_nov_p2_memo_maths                   memo              0
2025_nov_p1_exam_maths                   exam             11
2025_n

In [20]:
print("Notebook 03 v1 accepted")
print("Primary asset: data/processed/questions/question_structure.csv")

Notebook 03 v1 accepted
Primary asset: data/processed/questions/question_structure.csv
